# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score

# Cargo el dataset WBCD desde UCI (el archivo no trae encabezados)
df_wbcd = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data", header=None)

# Nombres de columnas: id, diagnosis y 30 características
# (10 medidas básicas x 3 versiones: mean, se y worst)
medidas = ["radius", "texture", "perimeter", "area", "smoothness",
           "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
columnas = ["id", "diagnosis"]
for sufijo in ["mean", "se", "worst"]:
    columnas += [m + "_" + sufijo for m in medidas]
df_wbcd.columns = columnas

print("Filas y columnas:", df_wbcd.shape)
df_wbcd.head()

In [ ]:
# Exploración rápida
print(df_wbcd["diagnosis"].value_counts())
print("\nValores nulos en total:", df_wbcd.isnull().sum().sum())

# Comparo las escalas de algunas variables (area_mean es enorme comparada con smoothness_mean)
df_wbcd[["radius_mean", "perimeter_mean", "area_mean", "smoothness_mean"]].describe().round(3)

## 2. Preprocesamiento

In [ ]:
# Convierto diagnosis a binario: M (maligno) = 1, B (benigno) = 0
df_wbcd["diagnosis"] = df_wbcd["diagnosis"].map({"M": 1, "B": 0})

# X = características (quito id y diagnosis), y = target
X = df_wbcd.drop(columns=["id", "diagnosis"])
y = df_wbcd["diagnosis"]

# Separo 80% train y 20% test (stratify para mantener la proporción de M y B)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 3. Escalamiento (Solo para k-NN)

In [ ]:
# StandardScaler: fit_transform solo en train, y solo transform en test
# (así no se filtra información del test al entrenamiento)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Modelado y Matrices de Confusión

In [ ]:
# k-NN con datos escalados
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# Árbol de decisión (no necesita escalamiento, uso X sin escalar)
arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)
y_pred_arbol = arbol.predict(X_test)

# Matrices de confusión: filas = real, columnas = predicho
# formato: [[VN, FP],
#           [FN, VP]]   (0 = benigno, 1 = maligno)
print("Matriz de confusión k-NN (escalado):")
print(confusion_matrix(y_test, y_pred_knn))
print("\nMatriz de confusión Árbol de Decisión:")
print(confusion_matrix(y_test, y_pred_arbol))

### Experimento: ¿qué pasa si no escalo los datos?
Para validarlo entreno k-NN **sin escalar** y el árbol **con datos escalados**, y comparo las cuatro combinaciones.

In [ ]:
# k-NN SIN escalar
knn_sin = KNeighborsClassifier(n_neighbors=5)
knn_sin.fit(X_train, y_train)
y_pred_knn_sin = knn_sin.predict(X_test)

# Árbol CON datos escalados
arbol_esc = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol_esc.fit(X_train_scaled, y_train)
y_pred_arbol_esc = arbol_esc.predict(X_test_scaled)

resultados = {
    "k-NN escalado": y_pred_knn,
    "k-NN SIN escalar": y_pred_knn_sin,
    "Árbol sin escalar": y_pred_arbol,
    "Árbol escalado": y_pred_arbol_esc,
}

for nombre, pred in resultados.items():
    cm = confusion_matrix(y_test, pred)
    print(nombre)
    print(cm)
    print(f"  Accuracy: {accuracy_score(y_test, pred):.3f} | "
          f"Recall (maligno): {recall_score(y_test, pred):.3f} | "
          f"Falsos negativos: {cm[1][0]}\n")

**Observaciones del experimento:**
- k-NN **sin escalar** empeoró: los falsos negativos subieron de 4 a 9 y el recall de los casos malignos bajó de 0.905 a 0.786.
- El Árbol de Decisión dio **exactamente la misma matriz de confusión** con y sin escalar.

## Evaluación Teórica / Decisión de Negocios

1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?
2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?

*Escribe tu respuesta aquí:*

**1. Error Tipo II (Falso Negativo)**

Un falso negativo es cuando el modelo dice que la paciente es **benigna (0)** pero en realidad tiene un tumor **maligno (1)**. En la matriz de confusión es la casilla de abajo a la izquierda (FN).

Cuesta vidas porque a esa paciente se le dice que está sana y no recibe tratamiento. El cáncer sigue avanzando y cuando se detecta después ya está en una etapa más grave, con menos probabilidades de sobrevivir. En cambio, un falso positivo (Error Tipo I) solo causa susto y estudios extra (por ejemplo una biopsia), y se corrige con más pruebas. Por eso en este problema el error más costoso es el FN, y no basta con mirar el accuracy: lo importante es el **recall** de la clase maligna. Por ejemplo, el k-NN escalado tiene 95.6% de accuracy, pero aun así dejó pasar 4 pacientes con cáncer, y el árbol dejó pasar 8.

**2. ¿Por qué el Árbol de Decisión no falló sin escalar?**

Porque el árbol no usa distancias. En cada nodo elige una sola variable y un umbral (por ejemplo `perimeter_worst <= t`) y mide qué tan bien separa las clases con Gini o entropía. Eso solo depende del **orden** de los valores de cada variable, no de su magnitud. Escalar con StandardScaler es una transformación lineal creciente, así que el orden no cambia: el árbol encuentra las mismas particiones, solo cambia el valor numérico del umbral. Lo comprobé en el experimento: las matrices del árbol escalado y sin escalar son idénticas.

El k-NN en cambio usa distancia euclidiana, y variables como `area_mean` (promedio ~655, hasta 2501) pesan muchísimo más que `smoothness_mean` (~0.09), entonces la distancia la dominan las variables de mayor escala y las demás casi no cuentan. Por eso sin escalar el k-NN pasó de 4 a 9 falsos negativos.